# Step 1: Freeze the 2017–2026 game sample

A game is kept when it is completed, both scores are present, and both teams have a row in the team box score (`team_box.parquet`). The season table counts the schedule before that filter.

Seasons 2006–2016 stay in the source files for the Elo burn-in in a later step. They are not part of this sample.

Outputs, written to `output/`:

- `sample_by_season.csv`: the paper's sample table
- `kept_game_ids.parquet`: the kept game ids, which later notebooks read
- `freeze_manifest.csv`: source file paths, modification dates, and rows read

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

DATA_FOLDER_NAME = "Data Source - Women's NCAA Basketball"


def find_project_dir(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / DATA_FOLDER_NAME).is_dir():
            return folder
    raise FileNotFoundError(f"No '{DATA_FOLDER_NAME}' folder above {start}")


PROJECT_DIR = find_project_dir(Path.cwd())
DATA_DIR = PROJECT_DIR / DATA_FOLDER_NAME
OUTPUT_DIR = PROJECT_DIR / "output"

GAMES_PATH = DATA_DIR / "games.parquet"
BOX_PATH = DATA_DIR / "team_box.parquet"

SEASON_MIN = 2017
SEASON_MAX = 2026
BURN_IN_MIN = 2006
POSTSEASON = 3

PROJECT_DIR

WindowsPath('C:/Users/jball/Downloads/Purdue University Work/Purdue 2nd Semester (Fall 2026)/Foundations of Statistical Machine Learning STAT 57900/Final Project')

## Load the schedule and the team box score

`games.parquet` has one row per game. `team_box.parquet` has one row per team per game. Only the columns this step needs are read.

In [2]:
GAME_COLUMNS = [
    "game_id",
    "season",
    "season_type",
    "game_date",
    "neutral_site",
    "home_score",
    "away_score",
    "status_type_completed",
]
BOX_COLUMNS = ["game_id", "team_id"]

games = pd.read_parquet(GAMES_PATH, columns=GAME_COLUMNS)
box = pd.read_parquet(BOX_PATH, columns=BOX_COLUMNS)

print(f"games.parquet: {len(games):,} rows, seasons {games['season'].min()}-{games['season'].max()}")
print(f"team_box.parquet: {len(box):,} rows")

games.parquet: 114,705 rows, seasons 2006-2026
team_box.parquet: 120,032 rows


## Apply the keep rule

A game needs box-score rows for two different teams, a completed status, and both scores.

In [3]:
team_counts = box.groupby("game_id")["team_id"].nunique()
both_teams = team_counts[team_counts >= 2].index

window = games.loc[games["season"].between(SEASON_MIN, SEASON_MAX)].copy()
window["has_box"] = window["game_id"].isin(both_teams)
window["is_postseason"] = window["season_type"].eq(POSTSEASON)
window["is_neutral"] = window["neutral_site"].fillna(False).astype(bool)
scores_present = window["home_score"].notna() & window["away_score"].notna()
completed = window["status_type_completed"].fillna(False).astype(bool)
window["kept"] = window["has_box"] & scores_present & completed

kept = (
    window.loc[window["kept"], ["game_id", "season"]]
    .sort_values(["season", "game_id"])
    .reset_index(drop=True)
)

print(f"scheduled {len(window):,}")
print(f"kept {len(kept):,}")

scheduled 55,154
kept 54,020


## Sample table by season

This is the paper's sample table. Season 2021 is the short season, and 2020 has no postseason games.

In [4]:
sample_by_season = (
    window.groupby("season", as_index=False)
    .agg(
        scheduled_games=("game_id", "size"),
        games_with_box_score=("has_box", "sum"),
        postseason_games=("is_postseason", "sum"),
        neutral_site_games=("is_neutral", "sum"),
        kept_games=("kept", "sum"),
    )
    .sort_values("season")
)
count_columns = sample_by_season.columns.drop("season")
sample_by_season[count_columns] = sample_by_season[count_columns].astype(int)
sample_by_season

,season,scheduled_games,games_with_box_score,postseason_games,neutral_site_games,kept_games
0,2017,5622,5242,141,560,5242
1,2018,5630,5283,141,568,5283
2,2019,5658,5366,141,510,5366
3,2020,5434,5426,0,385,5426
4,2021,3823,3823,117,390,3823
5,2022,5534,5512,142,505,5512
6,2023,5828,5815,142,557,5815
7,2024,5923,5898,145,554,5898
8,2025,5648,5626,129,477,5626
9,2026,6054,6029,145,573,6029


In [5]:
burn_in = games["season"].between(BURN_IN_MIN, SEASON_MIN - 1)
burn_in_completed = games.loc[burn_in, "status_type_completed"].fillna(False).astype(bool)
print(f"completed games available for Elo burn-in, {BURN_IN_MIN}-{SEASON_MIN - 1}: {int(burn_in_completed.sum()):,}")

completed games available for Elo burn-in, 2006-2016: 59,551


## Write the frozen sample

The manifest records each source file's modification date, so a later download that changes the data is visible.

In [6]:
def file_modified(path: Path) -> str:
    modified = datetime.fromtimestamp(path.stat().st_mtime, timezone.utc)
    return modified.strftime("%Y-%m-%d %H:%M:%S UTC")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table_path = OUTPUT_DIR / "sample_by_season.csv"
kept_path = OUTPUT_DIR / "kept_game_ids.parquet"
manifest_path = OUTPUT_DIR / "freeze_manifest.csv"

sample_by_season.to_csv(table_path, index=False)
kept.to_parquet(kept_path, index=False)

manifest = pd.DataFrame(
    [
        {
            "source": GAMES_PATH.name,
            "path": str(GAMES_PATH),
            "modified_utc": file_modified(GAMES_PATH),
            "rows_read": len(games),
        },
        {
            "source": BOX_PATH.name,
            "path": str(BOX_PATH),
            "modified_utc": file_modified(BOX_PATH),
            "rows_read": len(box),
        },
    ]
)
manifest.to_csv(manifest_path, index=False)

for path in [table_path, kept_path, manifest_path]:
    print(f"wrote {path}")

wrote C:\Users\jball\Downloads\Purdue University Work\Purdue 2nd Semester (Fall 2026)\Foundations of Statistical Machine Learning STAT 57900\Final Project\output\sample_by_season.csv
wrote C:\Users\jball\Downloads\Purdue University Work\Purdue 2nd Semester (Fall 2026)\Foundations of Statistical Machine Learning STAT 57900\Final Project\output\kept_game_ids.parquet
wrote C:\Users\jball\Downloads\Purdue University Work\Purdue 2nd Semester (Fall 2026)\Foundations of Statistical Machine Learning STAT 57900\Final Project\output\freeze_manifest.csv


# Step 2: One row per game

The team box has two rows for every game. From here on, the modeling table has one row per game, so those two rows are the two sides of a single observation.

The target is home score minus away score. `home_court` is 1 only when the site is not neutral, and `postseason` is 1 when `season_type` is 3. The row also carries the team ids, `game_date`, the season, and both rank fields.

Basketball games do not tie. The freeze above still contains 18 games, all on 7 January 2017, recorded as 0–0 in both files while the box score shows shot attempts. Those rows are recording errors, so this step drops them. The margin in the saved table is never 0.

In [7]:
ID_COLUMNS = [
    "game_id",
    "home_id",
    "away_id",
    "home_current_rank",
    "away_current_rank",
]
schedule = pd.read_parquet(GAMES_PATH, columns=ID_COLUMNS)

scores_differ = window["home_score"] != window["away_score"]
equal_score_games = int((window["kept"] & ~scores_differ).sum())
modeling_rows = window.loc[window["kept"] & scores_differ].copy()

game_table = modeling_rows.merge(schedule, on="game_id", how="left", validate="one_to_one")
game_table["margin"] = game_table["home_score"] - game_table["away_score"]
game_table["home_court"] = (~game_table["is_neutral"]).astype(int)
game_table["postseason"] = game_table["is_postseason"].astype(int)

game_table = game_table[
    [
        "game_id",
        "season",
        "game_date",
        "home_id",
        "away_id",
        "home_current_rank",
        "away_current_rank",
        "home_score",
        "away_score",
        "margin",
        "home_court",
        "postseason",
    ]
].sort_values(["season", "game_date", "game_id"]).reset_index(drop=True)

if not game_table["game_id"].is_unique:
    raise ValueError("game_table has more than one row for a game_id")
if (game_table["margin"] == 0).any():
    raise ValueError("game_table contains a margin of 0")
if not set(game_table["home_court"]).issubset({0, 1}):
    raise ValueError("home_court is not 0 or 1")
if not set(game_table["postseason"]).issubset({0, 1}):
    raise ValueError("postseason is not 0 or 1")

print(f"equal-score games dropped: {equal_score_games:,}")
print(f"game_table rows: {len(game_table):,}")
print(f"home-court games: {int(game_table['home_court'].sum()):,}")
print(f"postseason games: {int(game_table['postseason'].sum()):,}")
game_table.head()

equal-score games dropped: 18
game_table rows: 54,002
home-court games: 48,940
postseason games: 1,243


,game_id,season,game_date,home_id,away_id,home_current_rank,away_current_rank,home_score,away_score,margin,home_court,postseason
0,400905132,2017,2016-11-11,66,2540,NaN,NaN,77,60,17,1,0
1,400905150,2017,2016-11-11,239,2277,NaN,NaN,118,43,75,1,0
2,400911050,2017,2016-11-11,2306,2130,NaN,NaN,83,33,50,1,0
3,400911051,2017,2016-11-11,2628,2916,NaN,NaN,88,40,48,1,0
4,400911052,2017,2016-11-11,197,2032,NaN,NaN,76,34,42,1,0


In [8]:
game_table_path = OUTPUT_DIR / "game_table.parquet"
game_table.to_parquet(game_table_path, index=False)
print(f"wrote {game_table_path}")

wrote C:\Users\jball\Downloads\Purdue University Work\Purdue 2nd Semester (Fall 2026)\Foundations of Statistical Machine Learning STAT 57900\Final Project\output\game_table.parquet
